# E-Commerce Customer Segmentation: RFM Analysis

## Part 1: Data Cleaning & Pre-processing
To build an accurate RFM (Recency, Frequency, Monetary) model, we must ensure our data is clean and free of issues. E-commerce data inherently contains erorrs, such as guest checkouts, typos, and canceled orders which we must fix before working with the data.

In [57]:
import pandas as pd 
import numpy as np 

# Loading the csv file with the correct European encoding
df = pd.read_csv('data.csv', encoding='ISO-8859-1')

# Getting initial info from the data
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


### Handling "Ghost Shoppers" (Null CustomerIDs)
When a customers checks out as a "Guest" to buy an item such as a graphic hoodie, or collectible it generates revenue, but the customer cannot be tracked. Since the entire premise of an RFM model relies on tracking individual user behavior over time, any transaction missing a `CustomerID` is incompatible with our analysis and therefore must be dropped. 

In [58]:
# Instead of dropping and removing values from our original dataset, we will work with a copy instead
df_copy = df.copy()

# Dropping all rows with have NaN as their value in CustomerID 
df_copy = df_copy.dropna(subset=['CustomerID'])
 
# Sanity check to make sure all of the values were removed
df_copy.info()
df_copy[df_copy['CustomerID'].isna()]
df_copy.describe()

<class 'pandas.core.frame.DataFrame'>
Index: 406829 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    406829 non-null  object 
 1   StockCode    406829 non-null  object 
 2   Description  406829 non-null  object 
 3   Quantity     406829 non-null  int64  
 4   InvoiceDate  406829 non-null  object 
 5   UnitPrice    406829 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      406829 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 27.9+ MB


,Quantity,UnitPrice,CustomerID
count,406829.000000,406829.000000,406829.000000
mean,12.061303,3.460471,15287.690570
std,248.693370,69.315162,1713.600303
min,-80995.000000,0.000000,12346.000000
25%,2.000000,1.250000,13953.000000
50%,5.000000,1.950000,15152.000000
75%,12.000000,3.750000,16791.000000
max,80995.000000,38970.000000,18287.000000


### Filtering Cancellations and Returns
A quick review of the dataset reveals negative values within the `Quantity` column. In this specific dataset, negative values represent returns (i.e., a customer returning a puffer jacket) or canceled orders (i.e., a customer canceled a pre-order for a LEGO set).

If we were to include these negative values in our analysis, it would artificially decrease our `Monetary` calculations and misrepresent a customer's real purchasing history. We will filter the dataset to only inlcude successful, positive-quantity transactions.

In [59]:
# Keeping all rows where the 'Quantity' > 0
df_copy = df_copy[df_copy['Quantity'] > 0]

# Sanity check
df_copy.describe()

,Quantity,UnitPrice,CustomerID
count,397924.000000,397924.000000,397924.000000
mean,13.021823,3.116174,15294.315171
std,180.420210,22.096788,1713.169877
min,1.000000,0.000000,12346.000000
25%,2.000000,1.250000,13969.000000
50%,6.000000,1.950000,15159.000000
75%,12.000000,3.750000,16795.000000
max,80995.000000,8142.750000,18287.000000


### Feature Engineering: Total Sales
After getting rid of the negative values in our `Quantity` column we can now calculate the `M` (Monetary Value) in our RFM model, we need the total revenue generated per line item. We will create a new `Total_Sales` column by multiplying the number of items purchased (`Quantity`) by the cost per item (`UnitPrice`).

In [60]:
# Multiplying quantity by unit price to calculate total sales
df_copy['Total_Sales'] = df_copy['Quantity'] * df_copy['UnitPrice']

# Quick sanity check
df_copy.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Total_Sales
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,20.34


### Exporting the Cleaned Dataset
With all the 'Ghost Shoppers' removed, the returns filtered, and the monetary values calculated, the dataset is cleaned and ready to be analyzed. We will export this new cleaned dataset intoa. new CSV file and use it as a foundation for our SQL aggregations and cohort modeling. 

In [61]:
df_copy.to_csv('cleaned_ecommerce_data.csv', index = False)

## SQL Analytics (Building the RFM Model)
In e-commerce, the RFM Model is the standard for customer segmentation. It evaluates every shopper on three metrics: 

1. Recency: How many days has it been since their last purchase?
2. Frequency: How many distinct orders have they placed?
3. Monetary: How much total money have they spent?

By calculating these three numbers for every customer, we can instantly identify our VIPs (who buy high-value items like premium outerwear and specialized LEGO sets every week) versus your churn risks (who bought one graphic hoodie a year ago and never came back).

In [62]:
import sqlite3

# Creating a temp SQL database
conn = sqlite3.connect('ecommerce_segmentation.db')

# Loading cleaned dataframe into the SQL database
df_copy.to_sql('retail_ledger', conn, if_exists = 'replace', index = False)

397924

### Task 1: The High Rollers (Frequency & Monetary)
We will start by calculating the `F` and `M` of our RFM model. We want to find out how many distinct orders each customer has placed, and their total lifetime spend. 

In [63]:
query = """
SELECT
    CustomerID, 
    COUNT(DISTINCT InvoiceNo) AS total_num_of_purchases, 
    SUM(Total_Sales) AS total_revenue_generated
FROM 
    retail_ledger
GROUP BY 
    CustomerID
ORDER BY 
    SUM(Total_Sales) DESC
"""

pd.read_sql(query, conn)

,CustomerID,total_num_of_purchases,total_revenue_generated
0,14646.0,74,280206.02
1,18102.0,60,259657.30
2,17450.0,46,194550.79
3,16446.0,2,168472.50
4,14911.0,201,143825.06
...,...,...,...
4334,17956.0,1,12.75
4335,16454.0,2,6.90
4336,14792.0,1,6.20
4337,16738.0,1,3.75


### Task 2: Completing the RFM Base Table
We now need to inject the `Recency` metric to complete the foundation. We need to calculate exactly how many days have passed since each customer's last purchase.

In [64]:
query = """
SELECT
    CustomerID, 
    CAST(julianday('2011-12-10') - julianday(MAX(InvoiceDate)) AS INTEGER) AS Recency,
    COUNT(DISTINCT InvoiceNo) AS Frequency, 
    ROUND(SUM(Total_Sales), 2) AS Monetary
FROM 
    retail_ledger
GROUP BY 
    CustomerID
ORDER BY 
    Monetary DESC;
"""

pd.read_sql(query, conn)

,CustomerID,Recency,Frequency,Monetary
0,14646.0,None,74,280206.02
1,18102.0,None,60,259657.30
2,17450.0,None,46,194550.79
3,16446.0,None,2,168472.50
4,14911.0,None,201,143825.06
...,...,...,...,...
4334,17956.0,None,1,12.75
4335,16454.0,None,2,6.90
4336,14792.0,None,1,6.20
4337,16738.0,None,1,3.75


### Task 3: The Segmentation (CTE)

In [65]:
query = """
WITH RFM_Base AS (
    SELECT
        CustomerID, 
        CAST(julianday('2011-12-10') - julianday(MAX(InvoiceDate)) AS INTEGER) AS Recency,
        COUNT(DISTINCT InvoiceNo) AS Frequency, 
        ROUND(SUM(Total_Sales), 2) AS Monetary
    FROM 
        retail_ledger
    GROUP BY 
        CustomerID
)
SELECT 
    CustomerID,
    Recency,
    Frequency,
    Monetary,
    CASE
        WHEN Recency < 30 AND Frequency > 5 AND Monetary > 1000 THEN 'VIP'
        WHEN Recency >= 150 AND Frequency > 2 THEN 'At Risk (Previous Frequent Buyer)'
        WHEN Recency >= 200 AND Frequency = 1 THEN 'Lost (One-Time Buyer)'
        WHEN Recency < 30 AND Frequency = 1 THEN 'New Customer'
        ELSE 'Regular'
    END AS Customer_Segment
FROM 
    RFM_Base
ORDER BY 
    Monetary DESC;
"""

pd.read_sql(query, conn)

,CustomerID,Recency,Frequency,Monetary,Customer_Segment
0,14646.0,None,74,280206.02,Regular
1,18102.0,None,60,259657.30,Regular
2,17450.0,None,46,194550.79,Regular
3,16446.0,None,2,168472.50,Regular
4,14911.0,None,201,143825.06,Regular
...,...,...,...,...,...
4334,17956.0,None,1,12.75,Regular
4335,16454.0,None,2,6.90,Regular
4336,14792.0,None,1,6.20,Regular
4337,16738.0,None,1,3.75,Regular


In [66]:
# Run the final CTE query and save it to a dataframe
rfm_df = pd.read_sql(query, conn)

# Export the segmented data for Tableau
rfm_df.to_csv('rfm_segments.csv', index=False)

# Close the database connection
conn.close()